# Lesson 05: UNet++

Your first **architecture modification**. We keep the same encoder, data, loss and training recipe as Lesson 04, and change only the **decoder**.
Then we compare against your UNet baseline (**test MAE 0.0390, IoU 0.611**).

### The idea

In UNet, each skip connection joins an encoder feature straight to the decoder: `f1` (edges, texture) meets deep features (meaning) in one jump.
The UNet++ authors argued that this **semantic gap** is too large, so they filled the space between encoder and decoder with extra conv nodes.

```
UNet                                     UNet++
level 0   X00 ─────────────────► X04     X00 ─► X01 ─► X02 ─► X03 ─► X04
            ╲                   ↗          ╲   ↗ ╲   ↗ ╲   ↗ ╲   ↗
level 1     X10 ───────────► X13          X10 ─► X11 ─► X12 ─► X13
              ╲             ↗                ╲   ↗ ╲   ↗ ╲   ↗
level 2       X20 ─────► X22                X20 ─► X21 ─► X22
                ╲       ↗                      ╲   ↗ ╲   ↗
level 3         X30 ► X31                      X30 ─► X31
                  ╲   ↗                          ╲   ↗
level 4           X40                            X40
```

- `X[i][0]` = encoder features (`f1 … f5`), `X[4][0]` is the deepest.
- UNet only has the **diagonal** nodes: `X31, X22, X13, X04`.
- UNet++ fills in **every** node. Each one gets **all nodes to its left** at the same level (dense skips) plus the **upsampled node below-left**.

$$X^{i,j} = \text{conv}\Big(\big[\,X^{i,0}, X^{i,1}, \dots, X^{i,j-1},\ \text{up}(X^{i+1,j-1})\,\big]\Big)$$

**Deep supervision:** the top row `X01, X02, X03, X04` are all full-resolution predictions. We can put a head on each and train them all.
At test time you can use any of them. A shallow head is faster, which is the paper's "pruning" idea.

| Part | Topic |
|---|---|
| A | Build the node grid and print every node's inputs and shape |
| B | Cost: params and GPU memory vs the UNet decoder |
| C | Deep supervision loss |
| D | Train with the Lesson 04 recipe (same split, epochs, lr) |
| E | Test results, and evaluating each deep-supervision head (pruning) |
| F | **Compare with your UNet baseline**: the table, and the images where UNet++ helped or hurt most |

## Settings

In [ ]:
RUN_NAME = "resnet50_unetpp_ds"     # ✏️ new name for every experiment
QUICK = True                        # ✏️ True first, then False for the real run
BASELINE_RUN = "resnet50_unet_concat"  # your Lesson 04 run, to compare against

# Keep these the SAME as Lesson 04, so the only difference is the decoder (a fair comparison)
EPOCHS = 2 if QUICK else 25
BATCH_SIZE = 8
LR = 1e-4
NUM_WORKERS = 4
SEED = 42
PRETRAINED = True

# The UNet++ decoder knobs
LEVEL_CHANNELS = (32, 64, 128, 256)  # channels of the new nodes at levels 0..3
DEEP_SUPERVISION = True

In [ ]:
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from blocks import DoubleConv, count_params, show
from config import IMAGE_SIZE, OUTPUT_DIR, get_device
from data import CODDataset, make_splits
from decoders import SegModel, UNetDecoder
from encoders import ResNet50Encoder
from train_utils import append_result, bce_iou_loss, evaluate, train_one_epoch

device = get_device()
torch.manual_seed(SEED)
print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

CHECKPOINT_DIR = os.path.join(os.path.dirname(OUTPUT_DIR), "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## Part A: the node grid

The whole decoder is one class. Read `__init__` (which nodes exist and how many channels each receives)
and `forward` (the double loop over columns `j` and levels `i`). It's the formula above, line by line.

In [ ]:
class UNetPlusPlusDecoder(nn.Module):
    """
    UNet++ (Zhou et al., 2018): nested, dense skip connections.

    Nodes X[i][j]:  i = level (0 = highest resolution), j = column.
      X[i][0]  = encoder feature f(i+1)
      X[i][j]  = conv( concat( X[i][0], ..., X[i][j-1],  up(X[i+1][j-1]) ) )
    The final mask comes from X[0][L]. With deep supervision, every X[0][j] gets its own head.

    encoder_channels : the encoder's `.channels` (5 values)
    level_channels   : channels of the new nodes at levels 0..3 (like decoder_channels in UNet)
    deep_supervision : train with an output head on every X[0][j]
    """

    def __init__(self, encoder_channels, level_channels=(32, 64, 128, 256), deep_supervision=False, num_classes=1):
        super().__init__()
        e = list(encoder_channels)
        c = list(level_channels)
        self.L = len(e) - 1                    # 4 decoder columns for a 5-level encoder
        self.deep_supervision = deep_supervision

        self.nodes = nn.ModuleDict()
        for j in range(1, self.L + 1):
            for i in range(0, self.L + 1 - j):
                below = e[i + 1] if j == 1 else c[i + 1]          # channels of X[i+1][j-1]
                in_ch = e[i] + c[i] * (j - 1) + below             # X[i][0] + X[i][1..j-1] + up(below)
                self.nodes[f"x{i}_{j}"] = DoubleConv(in_ch, c[i])

        n_heads = self.L if deep_supervision else 1
        self.heads = nn.ModuleList([nn.Conv2d(c[0], num_classes, kernel_size=1) for _ in range(n_heads)])

    def forward(self, feats, out_size=None, all_heads=False):
        X = {(i, 0): f for i, f in enumerate(feats)}
        for j in range(1, self.L + 1):
            for i in range(0, self.L + 1 - j):
                target = X[(i, 0)].shape[-2:]
                up = F.interpolate(X[(i + 1, j - 1)], size=target, mode="bilinear", align_corners=False)
                inputs = [X[(i, k)] for k in range(j)] + [up]
                X[(i, j)] = self.nodes[f"x{i}_{j}"](torch.cat(inputs, dim=1))

        def finish(logits):
            if out_size is not None and logits.shape[-2:] != tuple(out_size):
                logits = F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)
            return logits

        if self.deep_supervision and (self.training or all_heads):
            return [finish(head(X[(0, j)])) for j, head in zip(range(1, self.L + 1), self.heads)]
        return finish(self.heads[-1](X[(0, self.L)]))

In [ ]:
# Run it on fake ResNet50 features and print every node
enc_ch = [64, 256, 512, 1024, 2048]
sizes = [176, 88, 44, 22, 11]
feats = [torch.randn(1, c, s, s) for c, s in zip(enc_ch, sizes)]

dec = UNetPlusPlusDecoder(enc_ch, level_channels=LEVEL_CHANNELS).eval()
print(f"{'node':<6}{'inputs':<40}{'in_ch':>7}{'out_ch':>7}   {'output size'}")
with torch.no_grad():
    X = {(i, 0): f for i, f in enumerate(feats)}
    for j in range(1, dec.L + 1):
        for i in range(0, dec.L + 1 - j):
            node = dec.nodes[f"x{i}_{j}"]
            up = F.interpolate(X[(i + 1, j - 1)], size=X[(i, 0)].shape[-2:], mode="bilinear", align_corners=False)
            X[(i, j)] = node(torch.cat([X[(i, k)] for k in range(j)] + [up], dim=1))
            inputs = ", ".join([f"X{i}{k}" for k in range(j)] + [f"up(X{i + 1}{j - 1})"])
            in_ch = node.block[0].conv.in_channels
            print(f"X{i}{j:<4}{inputs:<40}{in_ch:>7}{LEVEL_CHANNELS[i]:>7}   {tuple(X[(i, j)].shape[-2:])}")
    out = dec(feats, out_size=(IMAGE_SIZE, IMAGE_SIZE))
print(f"\nfinal output: {tuple(out.shape)}")

Compare this with the UNet decoder: UNet has **4 nodes** (the diagonal), and UNet++ has **10**.
The extra 6 are all the "in-between" nodes, and `X01`–`X03` run at the **highest resolution** (176×176), so they are the expensive ones.

> ✏️ **TRY IT**
> - Change `LEVEL_CHANNELS` to `(64, 64, 64, 64)` and re-run the cell. How do `in_ch` values change?
> - Which node has the most input channels, and why?

## Part B: the cost

More nodes means more compute and memory. Let's measure it against the Lesson 04 UNet decoder, with the same encoder and a real training step (forward + backward, batch 8, AMP).

In [ ]:
def build_unet():
    enc = ResNet50Encoder(pretrained=False)
    return SegModel(enc, UNetDecoder(enc.channels, decoder_channels=(256, 128, 64, 32)))


def build_unetpp(pretrained=False, deep_supervision=DEEP_SUPERVISION):
    enc = ResNet50Encoder(pretrained=pretrained)
    return SegModel(enc, UNetPlusPlusDecoder(enc.channels, level_channels=LEVEL_CHANNELS,
                                             deep_supervision=deep_supervision))


def train_step_cost(model, batch_size=BATCH_SIZE):
    model = model.to(device).train()
    x = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        out = model(x)
    out = out if isinstance(out, list) else [out]
    sum(o.float().mean() for o in out).backward()
    if device.type == "cuda":
        torch.cuda.synchronize()
        mem = f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    else:
        mem = "(CPU)"
    return mem, (time.perf_counter() - t0) * 1000


bs = BATCH_SIZE if device.type == "cuda" else 2
print(f"{'model':<22}{'decoder params':>16}{'memory':>12}{'time':>10}   (batch {bs})")
for name, builder in [("UNet decoder", build_unet), ("UNet++ decoder", build_unetpp)]:
    m = builder()
    train_step_cost(m, bs)                     # warm-up
    mem, ms = train_step_cost(m, bs)
    print(f"{name:<22}{count_params(m.decoder):>16,}{mem:>12}{ms:>8.0f}ms")
    del m

UNet++ has about a third more parameters (≈12.2M vs 9.0M decoder params). Most of them sit in the **deep** nodes (`X31`, `X21`),
which receive the huge 1024- and 2048-channel ResNet features, even though those nodes are small in H×W.
The **memory and time** cost comes from somewhere else: the top-row nodes `X01`–`X03` run at 176×176.
Parameters and memory are different things. Params depend on channels × kernel size; memory depends on **how big the feature maps are**.

## Part C: deep supervision

With `DEEP_SUPERVISION = True`, in training mode the model returns **4 predictions** (from `X01, X02, X03, X04`).
We compute the Lesson 04 loss on each one and average them. The shallow heads get a direct training signal,
which helps the early nodes learn useful features.

In `eval()` mode the model returns just the last head (`X04`), so `evaluate()` from Lesson 04 works unchanged.

In [ ]:
def deep_supervision_loss(outputs, target):
    """Average the BCE + IoU loss over every head (or just the one output if there is only one)."""
    if not isinstance(outputs, list):
        return bce_iou_loss(outputs, target)
    return sum(bce_iou_loss(o, target) for o in outputs) / len(outputs)


# Quick check on fake data
m = build_unetpp().to(device).train()
x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
y = (torch.rand(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=device) > 0.5).float()
outs = m(x)
print(f"train mode -> {len(outs) if isinstance(outs, list) else 1} outputs of shape {tuple(outs[0].shape if isinstance(outs, list) else outs.shape)}")
print(f"loss: {deep_supervision_loss(outs, y).item():.4f}")
m.eval()
with torch.no_grad():
    print(f"eval mode  -> one output of shape {tuple(m(x).shape)}")
del m

## Part D: train

Exactly the Lesson 04 recipe: `make_splits()` gives the **same** train/validation/test split (same seed),
with the same epochs, learning rate, batch size and augmentation. The only differences are the decoder and the deep-supervision loss.

In [ ]:
train_pairs, val_pairs, test_pairs = make_splits(seed=SEED, quick=QUICK)
print(f"train {len(train_pairs)}   val {len(val_pairs)}   test {len(test_pairs)}" + ("   (QUICK subset)" if QUICK else ""))

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                   persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(CODDataset(train_pairs, IMAGE_SIZE, train=True), shuffle=True, drop_last=True, **loader_args)
val_loader = DataLoader(CODDataset(val_pairs, IMAGE_SIZE), shuffle=False, **loader_args)
test_set = CODDataset(test_pairs, IMAGE_SIZE)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)

In [ ]:
torch.manual_seed(SEED)
model = build_unetpp(pretrained=PRETRAINED).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}.pth")
print(f"params: {count_params(model):,}")

history = {"train_loss": [], "val_mae": [], "val_iou": [], "val_dice": []}
best_mae, best_epoch = float("inf"), -1
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, loss_fn=deep_supervision_loss)
    val = evaluate(model, val_loader, device)
    lr = optimizer.param_groups[0]["lr"]
    scheduler.step()
    for k, v in [("train_loss", train_loss), ("val_mae", val["mae"]), ("val_iou", val["iou"]), ("val_dice", val["dice"])]:
        history[k].append(v)

    marker = ""
    if val["mae"] < best_mae:
        best_mae, best_epoch = val["mae"], epoch
        torch.save({"model": model.state_dict(), "epoch": epoch, "val": val,
                    "level_channels": LEVEL_CHANNELS, "deep_supervision": DEEP_SUPERVISION}, checkpoint_path)
        marker = "  ★ saved"
    print(f"epoch {epoch:>2}/{EPOCHS}  loss {train_loss:.4f}  |  val MAE {val['mae']:.4f}  IoU {val['iou']:.3f}  "
          f"Dice {val['dice']:.3f}  |  lr {lr:.1e}  {time.perf_counter() - t0:5.0f}s{marker}")

train_minutes = (time.perf_counter() - start) / 60
print(f"\nbest epoch {best_epoch}: val MAE {best_mae:.4f}   total {train_minutes:.1f} min")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, history["train_loss"], marker="o")
axes[0].set_title("train loss (avg over heads)")
axes[1].plot(epochs, history["val_mae"], marker="o", color="tab:red")
axes[1].axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch {best_epoch}")
axes[1].set_title("val MAE")
axes[1].legend()
axes[2].plot(epochs, history["val_iou"], marker="o", label="IoU")
axes[2].plot(epochs, history["val_dice"], marker="o", label="Dice")
axes[2].set_title("val IoU / Dice")
axes[2].legend()
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part E: test results, and pruning

Load the best checkpoint and evaluate on the test set.

With deep supervision we can also score **each head** on its own. `X01` uses only the first column of nodes, so it's much cheaper to run.
If it's nearly as good as `X04`, you could deploy a smaller, faster model. This is the UNet++ paper's "pruning".

In [ ]:
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model"])
print(f"loaded best checkpoint from epoch {ckpt['epoch']}")

test = evaluate(model, test_loader, device, per_image=True)
print(f"TEST ({len(test_set)} images):  MAE {test['mae']:.4f}   IoU {test['iou']:.3f}   Dice {test['dice']:.3f}")

In [ ]:
from train_utils import batch_metrics


@torch.no_grad()
def evaluate_heads(model, loader):
    """MAE / IoU for every deep-supervision head separately."""
    model.eval()
    scores = None
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            outs = model.decoder(model.encoder(images), out_size=images.shape[-2:], all_heads=True)
        scores = scores or [{"mae": [], "iou": []} for _ in outs]
        for s, logits in zip(scores, outs):
            m, i, _ = batch_metrics(logits, masks)
            s["mae"] += m
            s["iou"] += i
    return [(float(np.mean(s["mae"])), float(np.mean(s["iou"]))) for s in scores]


if DEEP_SUPERVISION:
    for j, (mae, iou_) in enumerate(evaluate_heads(model, test_loader), start=1):
        print(f"head X0{j}:  test MAE {mae:.4f}   IoU {iou_:.3f}")
else:
    print("DEEP_SUPERVISION is False, so there is only one head.")

## Part F: UNet++ vs your UNet baseline

A single average number hides a lot. Two models with the same MAE can fail on completely different images.
So we compare them **image by image**: where did UNet++ help most, and where did it hurt?

In [ ]:
def load_baseline():
    path = os.path.join(CHECKPOINT_DIR, f"{BASELINE_RUN}.pth")
    if not os.path.exists(path):
        print(f"Baseline checkpoint not found: {path}  (run Lesson 04 first)")
        return None
    ckpt = torch.load(path, map_location=device)
    enc = ResNet50Encoder(pretrained=False)  # the weights come from the checkpoint
    m = SegModel(enc, UNetDecoder(enc.channels, **ckpt["decoder_cfg"])).to(device)
    m.load_state_dict(ckpt["model"])
    return m


baseline = load_baseline()
if baseline is not None:
    base_test = evaluate(baseline, test_loader, device, per_image=True)
    print(f"{'model':<14}{'test MAE':>10}{'IoU':>8}{'Dice':>8}")
    print(f"{'UNet':<14}{base_test['mae']:>10.4f}{base_test['iou']:>8.3f}{base_test['dice']:>8.3f}")
    print(f"{'UNet++':<14}{test['mae']:>10.4f}{test['iou']:>8.3f}{test['dice']:>8.3f}")

    diff = np.array(test["per_image_iou"]) - np.array(base_test["per_image_iou"])
    print(f"\nUNet++ better on {np.sum(diff > 0.05)} images, worse on {np.sum(diff < -0.05)} (by more than 0.05 IoU)")

In [ ]:
@torch.no_grad()
def predict(m, index):
    image_t, mask_t = test_set[index]
    m.eval()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        logits = m(image_t[None].to(device))
    return torch.sigmoid(logits.float())[0, 0].cpu().numpy(), mask_t[0].numpy()


def outline(image, gt, prob):
    out = image.copy()
    for mk, color in [(gt > 0.5, (0, 255, 0)), (prob > 0.5, (0, 0, 255))]:
        contours, _ = cv2.findContours(mk.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, color, 2)
    return out


if baseline is not None:
    order = np.argsort(diff)
    for label, idxs in [("UNet++ HELPED most", order[::-1][:3]), ("UNet++ HURT most", order[:3])]:
        tiles, titles = [], []
        for i in idxs:
            i = int(i)
            image, _ = test_set.load(i)
            image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
            p_base, gt = predict(baseline, i)
            p_pp, _ = predict(model, i)
            name = os.path.basename(test_pairs[i][0]).replace("COD10K-CAM-", "")[:24]
            tiles += [outline(image, gt, p_base), outline(image, gt, p_pp)]
            titles += [f"UNet {base_test['per_image_iou'][i]:.2f}  {name}", f"UNet++ {test['per_image_iou'][i]:.2f}"]
        print(label)
        show(tiles, titles, cols=4, size=3.5)

Green = GT, red = prediction. Look for a **pattern**: are the helped images the ones with thin parts, small objects or cluttered backgrounds?
That kind of observation is where your own architecture ideas will come from.

## Log the run

In [ ]:
log_path = os.path.join(OUTPUT_DIR, "results.csv")
append_result(log_path, {
    "run": RUN_NAME, "quick": QUICK, "epochs": EPOCHS, "best_epoch": best_epoch, "batch": BATCH_SIZE, "lr": LR,
    "pretrained": PRETRAINED, "decoder": f"UNet++ {LEVEL_CHANNELS} ds={DEEP_SUPERVISION}",
    "params_M": round(count_params(model) / 1e6, 2),
    "val_mae": round(best_mae, 4), "test_mae": round(test["mae"], 4), "test_iou": round(test["iou"], 4),
    "test_dice": round(test["dice"], 4), "train_min": round(train_minutes, 1),
})
with open(log_path) as f:
    print(f.read())

> ✏️ **TRY IT** (new `RUN_NAME` each time):
> 1. `DEEP_SUPERVISION = False`: is it the extra nodes or the deep supervision that helps?
> 2. `LEVEL_CHANNELS = (64, 64, 64, 64)`: wider top level, narrower deep levels.
> 3. If a shallow head (e.g. `X02`) scored almost as well as `X04` in Part E, how much faster is it? (Hint: the Part B cost function.)
>
> **How to judge a difference:** the gap between two runs has to be bigger than the noise.
> Training the *same* model twice with a different `SEED` can shift MAE by a few thousandths.
> If you want to be sure, run both models with 2–3 seeds. That's what a thesis experiment needs.

---
## Summary

- UNet++ = UNet + **nested dense skip nodes** that bridge the semantic gap between encoder and decoder features, plus optional **deep supervision**.
- It changes **only the decoder**. The encoder, data, loss and training recipe stay the same, which is what makes the comparison fair.
- Fewer params doesn't mean cheaper: high-resolution nodes cost **memory and time**.
- Compare models **per image**, not just by the average.

**Next lesson (06): attention.** SE and CBAM modules that let the network decide *which channels* and *where* to look, added to the skip connections.
That's the step from "plain" decoders towards COD-specific designs like SINet.